# RAG with LangChain and Hugging Face

A Retrieval-Augmented Generation (RAG) system using LangChain, Hugging Face embeddings,
FAISS, and a Hugging Face question-answering model.

Meets the exercise requirements:
- Loads **`databricks/databricks-dolly-15k`** via `HuggingFaceDatasetLoader`
- Splits text with `chunk_size=1000`, `chunk_overlap=150`
- Uses the **`Intel/dynamic_tinybert`** QA model
- Orchestrates retrieval + answering with LangChain's **`RetrievalQA`** chain

### How to run (read this first)
1. Run the install cell **once**.
2. `Runtime -> Restart session` (or run the kill cell), wait to reconnect.
3. Run every cell from "Imports" downward. **Do not run the install cell again** in the same session.

## 1. Install (run ONCE, then restart)

We pin **langchain to 0.2.16** so that `RetrievalQA` exists at `langchain.chains` (it was
removed in langchain 1.x). The Hugging Face stack (transformers/accelerate/numpy) is left
unpinned so it matches Colab's pre-installed, mutually-consistent versions.

In [1]:
# transformers pinned to 4.x: v5 removed the "question-answering" pipeline task.
# langchain pinned to 0.2.16: v1.x removed RetrievalQA.
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1"
!pip install -q "langchain==0.2.16" "langchain-community==0.2.16" "langchain-huggingface==0.0.3" "langchain-text-splitters==0.2.4"
!pip install -q sentence-transformers faiss-cpu datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

## 2. Restart the session

Run this cell once after installing. The session will say it crashed/disconnected — that is
expected. Wait for it to reconnect, then continue from the next cell. **Do not re-run the
install cell.**

In [ ]:
import os
os.kill(os.getpid(), 9)

## 3. Imports

In [1]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

import transformers, langchain
print("transformers:", transformers.__version__)
print("langchain:", langchain.__version__)
print("Imports OK")

transformers: 4.44.2
langchain: 0.2.16
Imports OK


## 4. Load the databricks-dolly-15k dataset

We use `HuggingFaceDatasetLoader` to load the real `databricks/databricks-dolly-15k`
dataset from the Hugging Face Hub. The `context` column holds the reference passages,
so it is used as the `page_content_column`.

The full dataset has ~15k records; many have an empty `context`. We drop the empty ones and
keep a manageable subset so the FAISS index builds quickly on CPU. The loader still loads the
full real dataset as required — set `SUBSET = None` to index everything.

In [2]:
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()
print("Total documents loaded:", len(data))

# Keep only documents that actually have context text
data = [d for d in data if d.page_content and d.page_content.strip()]
print("Documents with non-empty context:", len(data))

# Subset for a fast CPU demo (set to None to use all)
SUBSET = 500
if SUBSET is not None:
    data = data[:SUBSET]
print("Documents used in this run:", len(data))
print(data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Total documents loaded: 15011
Documents with non-empty context: 15011
Documents used in this run: 500
page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


## 5. Split documents into chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = text_splitter.split_documents(data)
print("Number of chunks:", len(docs))
print(docs[0])

Number of chunks: 567
page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


## 6. Create embeddings

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": False}
)

test_embedding = embeddings.embed_query("This is a test document.")
print(test_embedding[:3])
print("Embedding size:", len(test_embedding))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[-0.038338519632816315, 0.1234646886587143, -0.028642984107136726]
Embedding size: 384


## 7. Build the FAISS vector store and retriever

In [5]:
db = FAISS.from_documents(docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})
print("Retriever created successfully")

Retriever created successfully


## 8. Load the Intel/dynamic_tinybert QA model

We load `Intel/dynamic_tinybert` into a Hugging Face `question-answering` pipeline and wrap it
in `HuggingFacePipeline` so it can act as the LLM inside a `RetrievalQA` chain.

In [6]:
model_name = "Intel/dynamic_tinybert"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Extractive QA pipeline (needs separate question + context)
question_answerer = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer
)

# --- Adapter LLM ---
# RetrievalQA sends ONE combined prompt string to the LLM, but an extractive QA
# pipeline needs (question, context) separately. This small LLM wrapper takes the
# prompt RetrievalQA builds, splits the question off the end, treats the rest as
# context, and calls the QA pipeline correctly.
from langchain_core.language_models.llms import LLM
from typing import Any, List, Optional

class TinyBertQALLM(LLM):
    @property
    def _llm_type(self) -> str:
        return "tinybert_qa"

    def _call(self, prompt: str, stop: Optional[List[str]] = None,
              run_manager: Any = None, **kwargs: Any) -> str:
        # RetrievalQA's default prompt ends with: "Question: <q>\nHelpful Answer:"
        question = prompt
        context = prompt
        if "Question:" in prompt:
            before, after = prompt.rsplit("Question:", 1)
            context = before
            question = after.split("Helpful Answer:")[0].strip()
        result = question_answerer(question=question, context=context)
        return result["answer"]

llm = TinyBertQALLM()
print("QA model loaded and wrapped (adapter) for RetrievalQA")

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

QA model loaded and wrapped (adapter) for RetrievalQA


## 9. Build the RetrievalQA chain

`RetrievalQA` handles the full RAG flow: retrieve the relevant chunks, pass them as context to
the QA model, and return the answer.

In [7]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)
print("RetrievalQA chain created successfully")

RetrievalQA chain created successfully


## 10. Ask questions

The answers come from the Dolly dataset content, so good questions are ones the dataset can
answer (general knowledge / Wikipedia-style topics).

In [8]:
def ask(question):
    result = qa_chain.invoke({"query": question})
    print("Question:", question)
    print("Answer:", result["result"])
    print("\nTop source passage:")
    print(result["source_documents"][0].page_content[:300], "...")
    print("-" * 60)
    return result

_ = ask("What is the capital of France?")

Question: What is the capital of France?
Answer: Ponferrada

Top source passage:
"Congosto (Spanish pronunciation: [ko\u014b\u02c8\u0261osto]) is a village and municipality located in the region of El Bierzo (province of Le\u00f3n, Castile and Le\u00f3n, Spain) . It is located near to Ponferrada, the capital of the region. The village of Congosto has about 350 inhabitants.\n\nIt ...
------------------------------------------------------------


In [9]:
_ = ask("Who wrote Hamlet?")

Question: Who wrote Hamlet?
Answer: Johnny Mercer

Top source passage:
"\"The Fox in the Attic\" was originally published in 1961 by Chatto & Windus: London as v. 1 of The Human Predicament trilogy, and then in the United States by Harper & Brothers: New York. This was 23 years after Hughes's previous novel, In Hazard: A Sea Story, and 33 years after A High Wind in Jam ...
------------------------------------------------------------
